# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *xx*  
**Kaggle challenge:** *xx* (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "*xx*"  

**Author 1 (SCIPER):** *Student Name 1 (xxxxx)*  
**Author 2 (SCIPER):** *Student Name 2 (xxxxx)*  
**Author 3 (SCIPER):** *Student Name 3 (xxxxx)*  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

### Imports :

In [1]:
import os
import sys 
assert (sys.version_info.major == 3) and (sys.version_info.minor == 9)

In [2]:
# if not done yet, install the required packages :
#!pip install -r requirements.txt

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import mean_absolute_error, r2_score
from tqdm import tqdm

### Paths :

In [4]:
dataset_dir = "dataset_project_iapr2025"
images_train_dir = os.path.join(dataset_dir, "train")
train_csv_path = os.path.join(dataset_dir, "train.csv")

In [5]:
dataframe_train = pd.read_csv(train_csv_path)
dataframe_train["filename"] = dataframe_train["id"].apply(lambda x: f"L{x:07d}.JPG")
print(dataframe_train)

         id  Jelly White  Jelly Milk  Jelly Black  Amandina  Crème brulée  \
0   1000756            2           0            0         0             0   
1   1000763            2           3            3         0             0   
2   1000765            0           0            0         0             0   
3   1000768            0           0            0         0             0   
4   1000772            3           2            1         0             0   
..      ...          ...         ...          ...       ...           ...   
85  1010031            0           0            0         0             0   
86  1010032            0           0            0         0             0   
87  1010035            0           0            0         0             0   
88  1010041            0           0            0         2             0   
89  1010045            1           1            1         1             1   

    Triangolo  Tentation noir  Comtesse  Noblesse  Noir authentique  \
0   

In [6]:
def check_image_sizes(image_dir):
    sizes = set()  # set() only keeps unique values
    
    for filename in os.listdir(image_dir):
        if filename.endswith(".JPG"):
            img_path = os.path.join(image_dir, filename)
            image = Image.open(img_path)
            sizes.add(image.size)  # Adds the size (width, height) to the set
    
    if len(sizes) > 1:
        print("Images are of different sizes:")
        print(sizes)
    else:
        print("Every image has the same size :", sizes)

check_image_sizes(images_train_dir)

Every image has the same size : {(6000, 4000)}


### Dataset :

In [7]:
class ChocolateDataset(Dataset): # inherits from Dataset class in PyTorch
    def __init__(self, dataframe, images_dir, transform=None):
        self.dataframe = dataframe # dataframe = raw data
        self.images_dir = images_dir # directory where images are stored
        self.transform = transform # transformations to be applied to images
        self.labels = dataframe.iloc[:, 1:-1].values # all columns except the first and last one (id and filename)
        self.filenames = dataframe["filename"].values

    def __len__(self):
        return len(self.dataframe) # number of samples in the dataset

    def __getitem__(self, idx): # will be called each time we iterate over the dataset in a dataloader
        img_name = os.path.join(self.images_dir, self.filenames[idx])
        image = Image.open(img_name).convert("RGB")
        if self.transform:
            image = self.transform(image) # will be applied each time we iterate over the dataset in a dataloader
        
        label = torch.tensor(self.labels[idx], dtype=torch.float32)  # 13 classes
        return image, label


In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Create the dataset and dataloader WITHOUT normalization
train_dataset = ChocolateDataset(dataframe_train, images_train_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True)

In [9]:
def calculate_mean_std(dataloader):
    means = []
    stds = []
    for images, _ in dataloader:
        # Calculate mean and std for each batch
        means.append(torch.mean(images, dim=[0, 2, 3])) # size of the image is (batch_size, channels, height, width) => mean for each channel (RGB), output is (3,)
        stds.append(torch.std(images, dim=[0, 2, 3])) # std for each channel (RGB)
    
    # Mean and std are calculated across all batches
    mean = torch.mean(torch.stack(means), dim=0).numpy()
    std = torch.mean(torch.stack(stds), dim=0).numpy()
    
    return mean, std

# Calculate mean and std with the dataloader
mean, std = calculate_mean_std(train_loader)

print(f"Mean: {mean}")
print(f"Std: {std}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


# Recreate the dataset and dataloader WITH normalization
train_dataset = ChocolateDataset(dataframe_train, images_train_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True)

Mean: [0.6840921  0.6594279  0.65269643]
Std: [0.16892572 0.16662638 0.19099812]


### CNN : MAYBE WE NEED TO USE ATTENTION GATED POOLING LIKE LAB 3 ????????????????????????

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class ChocolateCounterCNN(nn.Module):
    def __init__(self):
        super(ChocolateCounterCNN, self).__init__() # call the constructor of the parent class nn.Module
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), # 3 channels (RGB) -> 16 channels
            nn.ReLU(),
            nn.MaxPool2d(2),  # from 224x224 pixels to 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 16 channels -> 32 channels
            nn.ReLU(),
            nn.MaxPool2d(2),  # from 112x112 pixels to 56x56

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 channels -> 64 channels
            nn.ReLU(),
            nn.MaxPool2d(2),  # from 56x56 pixels to 28x28

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64 channels -> 128 channels
            nn.ReLU(),
            nn.MaxPool2d(2), # from 28x28 pixels to 14x14
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(), # flatten the output of the conv layers
            nn.Linear(128 * 14 * 14, 512), # 128 channels * (14*14) pixels to 512 neurons
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 13)  # 13 types of chocolates
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

## From practise 3 DL :

In [12]:
def mae(preds, target):
    return mean_absolute_error(target, preds)

def r2(preds, target):
    return r2_score(target, preds)


def train_epoch(model, optimizer, criterion, metrics, train_loader, device):
    
    # we activate the training state for the model
    model.train()

    # initialize epoch loss and metrics
    epoch_loss = 0
    epoch_metrics = dict(zip(metrics.keys(), torch.zeros(len(metrics))))

    # we iterate over the batches
    for batch_num, (x_batch, y_batch) in tqdm(enumerate(train_loader)):
        data = x_batch.to(device) # (batch_size, channels, height, width)
        target = y_batch.to(device) #Le terme device dans PyTorch fait référence à l'appareil (CPU ou GPU) sur lequel les tenseurs et les modèles sont stockés et exécutés.

        # Clean previously computed gradients
        optimizer.zero_grad()
        
        # Forward pass
        output = model(data) #calcule les prédictions du modèle. (scores bruts, pas encore les argmax)

        # Compute loss
        loss = criterion(output, target)

        # Compute gradients and optimize the model parameters
        loss.backward() #calcule les gradients
        optimizer.step() #met à jour les paramètres du modèle.
        
        '''
        Compute predictions and metrics
        Store predictions in variable preds
        No gradients should be propagated at this step
        '''
        with torch.no_grad(): #Cela garantit que PyTorch n'enregistre pas les opérations associées à torch.max() dans le graphe computationnel, ce qui évite une consommation mémoire excessive et optimise l'entraînement.
            preds = output.detach().cpu()
            target = target.detach().cpu()
            for k in epoch_metrics.keys():
                epoch_metrics[k] += metrics[k](preds, target)


        # log loss statistics
        epoch_loss += loss.item()

    # for the epoch loss, we take the average of the losses computed over the mini-batches
    epoch_loss /= len(train_loader)

    # for the epoch loss, we compute the average of the metrics over the mini-batches
    for k in epoch_metrics.keys():
          epoch_metrics[k] /= len(train_loader)

    clear_output() #clean the prints from previous epochs
    print('train Loss: {:.4f}, '.format(epoch_loss),
          ', '.join(['{}: {:.4f}'.format(k, epoch_metrics[k]) for k in epoch_metrics.keys()]))

    return epoch_loss,  epoch_metrics


In [13]:
def evaluate(model, criterion, metrics, test_loader, device):

    # we activate the evaluation state for the model
    model.eval()

    # initialize epoch loss and metrics
    epoch_loss = 0
    epoch_metrics = dict(zip(metrics.keys(), torch.zeros(len(metrics))))

    for batch_num, (x_batch, y_batch) in enumerate(test_loader):
        with torch.no_grad(): # since we do not update the model, we do not compute gradients
            data = x_batch.to(device)
            target = y_batch.to(device)

            outputs = model(data)
            loss = criterion(outputs, target)
            _, preds = torch.max(outputs, 1)
            for k in epoch_metrics.keys():
                epoch_metrics[k] += metrics[k](preds.cpu(), target.cpu())

            epoch_loss += loss.item()
    epoch_loss /= len(test_loader)
    for k in epoch_metrics.keys():
          epoch_metrics[k] /= len(test_loader)

    print('eval Loss: {:.4f}, '.format(epoch_loss),
          ', '.join(['{}: {:.4f}'.format(k, epoch_metrics[k]) for k in epoch_metrics.keys()]))
    
    return epoch_loss,  epoch_metrics

# Liste des noms de chocolat dans l'ordre (doit être le même que dans les labels)
chocolate_names = [
    "Jelly White", "Jelly Milk", "Jelly Black",
    "Amandina", "Crème brulée", "Triangolo",
    "Tentation noir", "Comtesse", "Noblesse",
    "Noir authentique", "Passion au lait", "Arabia",
    "Stracciatella"
]

def generate_predictions_csv(model, test_loader, device, output_path="submission.csv"):
    model.eval()
    predictions = []
    image_ids = []

    with torch.no_grad():
        for x_batch in tqdm(test_loader):

            x_batch = x_batch.to(device)
            outputs = model(x_batch)
            outputs = torch.round(outputs).cpu().numpy().astype(int)

            # Récupération des noms ou IDs d'images depuis le dataset
            # Ici, on suppose que test_loader.dataset.samples[i][0] contient le chemin
            batch_size = outputs.shape[0]
            for i in range(batch_size):
                # Exemple : extrait "1000756" de ".../1000756.jpg"
                path = test_loader.dataset.samples[len(image_ids)][0]
                image_id = int(path.split("/")[-1].split(".")[0])
                image_ids.append(image_id)
                predictions.append(outputs[i])

    # Création du DataFrame
    df = pd.DataFrame(predictions, columns=chocolate_names)
    df.insert(0, "id", image_ids)
    df.to_csv(output_path, index=False)
    print(f"✅ CSV enregistré : {output_path}")

In [14]:
def plot_training(train_loss, metrics_names, train_metrics_logs):
    fig, ax = plt.subplots(1, len(metrics_names) + 1, figsize=((len(metrics_names) + 1) * 5, 5))

    ax[0].plot(train_loss, c='blue', label='train')
    ax[0].set_title('Loss')
    ax[0].set_xlabel('epoch')
    ax[0].legend()

    for i in range(len(metrics_names)):
        ax[i + 1].plot(train_metrics_logs[i], c='blue', label='train')
        ax[i + 1].set_title(metrics_names[i])
        ax[i + 1].set_xlabel('epoch')
        ax[i + 1].legend()

    plt.show()

def update_metrics_log(metrics_names, metrics_log, new_metrics_dict):
    '''
    - metrics_names: the keys/names of the logged metrics
    - metrics_log: existing metrics log that will be updated
    - new_metrics_dict: epoch_metrics output from train_epoch and evaluate functions
    '''
    for i in range(len(metrics_names)):
        curr_metric_name = metrics_names[i]
        metrics_log[i].append(new_metrics_dict[curr_metric_name])
    return metrics_log

In [15]:
def train_cycle(model, optimizer, criterion, metrics, train_loader, n_epochs, device):
    train_loss_log = []
    metrics_names = list(metrics.keys())
    train_metrics_log = [[] for _ in range(len(metrics))]

    for epoch in range(n_epochs):
        print(f"Epoch {epoch+1} / {n_epochs}")
        train_loss, train_metrics = train_epoch(model, optimizer, criterion, metrics, train_loader, device)

        train_loss_log.append(train_loss)
        train_metrics_log = update_metrics_log(metrics_names, train_metrics_log, train_metrics)

        plot_training(train_loss_log, metrics_names, train_metrics_log)

    return train_metrics_log


In [ ]:
torch.manual_seed(42)

model = ChocolateCounterCNN()

metrics = {'MAE': mae, 'R²': r2}


#optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # or lr=1e-3

criterion = nn.MSELoss()  # or nn.L1Loss() if we want to be more robust to outliers

N_EPOCHS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

model.to(device)
criterion.to(device)

train_metrics_log = train_cycle(model, optimizer, criterion, metrics, train_loader=train_loader, n_epochs=N_EPOCHS, device=device)

# save model weights
results_models_weights_dir = 'models_weights/'
if not os.path.exists(results_models_weights_dir):
    os.mkdir(results_models_weights_dir)
torch.save(model.state_dict(), results_models_weights_dir + 'base_model.pth')

Using cpu
Epoch 1 / 30


0it [00:00, ?it/s]